# Sneaker Doctor · Entrenamiento en Google Colab

**Antes de empezar**
1. Menú **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4 → Guardar**.
2. Ícono de **llave (Secrets)** en la barra izquierda → agrega tu token de Kaggle y activa *Acceso del notebook*:
   - Si tu token es un texto largo (API token nuevo): nombre `KAGGLE_API_TOKEN`.
   - Si tienes el archivo `kaggle.json`: dos secrets, `KAGGLE_USERNAME` y `KAGGLE_KEY`, con los valores de ese archivo.
3. Ejecuta las celdas en orden con el botón ▶. Las celdas marcadas *(opcional)* son para sumar fotos reales de Clean Lab.

In [ ]:
# 1. Sube sneaker-doctor.zip (está en tu carpeta del bootcamp)
from google.colab import files
files.upload()

In [ ]:
# 2. Prepara el proyecto y el token de Kaggle
import os
from google.colab import userdata

def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

token, user, key = secret('KAGGLE_API_TOKEN'), secret('KAGGLE_USERNAME'), secret('KAGGLE_KEY')
if token:
    os.environ['KAGGLE_API_TOKEN'] = token
elif user and key:
    os.environ['KAGGLE_USERNAME'], os.environ['KAGGLE_KEY'] = user, key
else:
    raise SystemExit('No encontré el token de Kaggle en Secrets (ícono de llave). Revisa el paso 2 de arriba.')

!unzip -q -o sneaker-doctor.zip -d /content
%cd /content/sneaker-doctor
!pip -q install -U kaggle python-dotenv
print('Listo')

In [ ]:
# 3. Descarga UT Zappos50K desde Kaggle (uso académico, no comercial). Tarda unos minutos.
!kaggle datasets download -d aryashah2k/large-shoe-dataset-ut-zappos50k -p data/raw/zappos --unzip
!du -sh data/raw/zappos

In [ ]:
# 4. Etiqueta las fotos de Kaggle en gamuza, cuero, lona y malla
!python scripts/prepare_zappos.py --max-per-class 1500

In [ ]:
# 5. (opcional) Fotos reales de Clean Lab: sube fotos_cleanlab.zip con carpetas gamuza/, cuero/, lona/, malla/
#    y nombres tipo Z0001_1.jpg (mismo código = misma zapatilla)
files.upload()
!unzip -q -o fotos_cleanlab.zip -d data/raw/fotos_cleanlab
!python scripts/prepare_cleanlab.py --photos-dir data/raw/fotos_cleanlab

In [ ]:
# 6. Entrena: capa final + fine-tuning de las últimas capas con data augmentation (con GPU, unos 15-25 minutos)
#    Si hiciste el paso 5, se suman las fotos de Clean Lab automáticamente.
extra = '--extra data/processed/cleanlab.csv' if os.path.exists('data/processed/cleanlab.csv') else ''
!python scripts/train.py --finetune-epochs 6 {extra}

In [ ]:
# 7. Descarga el modelo entrenado y la lista de errores (para revisar en qué falla)
!cd models && zip -q ../modelo.zip sneaker_doctor.keras labels.json metrics.json errores_prueba.csv
files.download('modelo.zip')